# Thai Election OCR Pipeline (Qwen2.5-VL)
## Super AI Engineer Season 6 - OCR Competition 2569

Uses **Qwen2.5-VL-3B-Instruct** running locally on Colab GPU.
- No API rate limits
- Understands Thai + table structure
- Outputs structured JSON directly

**Runtime: GPU (T4)** — `Runtime > Change runtime type > T4 GPU`

In [ ]:
# === Cell 1: Install Dependencies ===
!pip install -q git+https://github.com/huggingface/transformers accelerate
!pip install -q bitsandbytes qwen-vl-utils Pillow rapidfuzz tqdm 2>/dev/null
print('Install complete.')

In [ ]:
# === Cell 2: Kaggle Download ===
import os, json

os.makedirs('/root/.kaggle', exist_ok=True)
from google.colab import userdata
username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': username, 'key': key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print(f'Kaggle credentials set: {username}')

COMPETITION = 'super-ai-engineer-season-6-ocr-2569'
!kaggle competitions download -c {COMPETITION} -p /content/ 2>&1

import glob as glob_module
for zf in glob_module.glob('/content/*.zip'):
    print(f'Extracting: {zf}')
    !unzip -qo "{zf}" -d /content/
print('Done.')

In [ ]:
# === Cell 3: Imports & Configuration ===
import os, re, json, time
import pandas as pd
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from tqdm.notebook import tqdm
from rapidfuzz import fuzz
import glob as glob_module

# Auto-discover paths
found = glob_module.glob('/content/**/images', recursive=True)
IMAGE_DIR = Path(found[0]) if found else Path('/content/data/images')
DATA_DIR = IMAGE_DIR.parent
SAMPLE_LABELS_DIR = DATA_DIR / 'sample_labels'

found_tmpl = glob_module.glob('/content/**/submission_template.csv', recursive=True)
SUBMISSION_TEMPLATE = Path(found_tmpl[0]) if found_tmpl else DATA_DIR / 'submission_template.csv'

OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / 'ocr_results.json'
PROGRESS_FILE = OUTPUT_DIR / 'progress.txt'

images = sorted(IMAGE_DIR.glob('*.png'))
print(f'Images: {len(images)}')
print(f'Template: {SUBMISSION_TEMPLATE} (exists={SUBMISSION_TEMPLATE.exists()})')
print(f'Sample labels: {SAMPLE_LABELS_DIR.exists()}')

In [ ]:
# === Cell 4: Load Template & Group Documents ===
template_df = pd.read_csv(SUBMISSION_TEMPLATE)
print(f'Template: {template_df.shape}')
print(template_df.head())

def group_documents(image_dir):
    docs = defaultdict(list)
    pattern = re.compile(r'^(.+?)(?:_page(\d+))?\.png$')
    for p in sorted(image_dir.glob('*.png')):
        m = pattern.match(p.name)
        if m:
            doc_id = m.group(1)
            page = int(m.group(2)) if m.group(2) else 1
            docs[doc_id].append((page, str(p)))
    for d in docs:
        docs[d].sort()
    return dict(docs)

documents = group_documents(IMAGE_DIR)
print(f'Documents: {len(documents)}')

# Template lookup
template_doc_rows = template_df.groupby('doc_id').size().to_dict()
doc_expected = defaultdict(list)
for _, row in template_df.iterrows():
    doc_expected[row['doc_id']].append({
        'id': row['id'],
        'row_num': row['row_num'],
        'party_name': row['party_name'],
    })
for d in doc_expected:
    doc_expected[d].sort(key=lambda x: x['row_num'])

doc_types = {}
for d in documents:
    doc_types[d] = 'partylist' if 'party_list' in d.lower() else 'constituency'

c_count = sum(1 for v in doc_types.values() if v == 'constituency')
p_count = sum(1 for v in doc_types.values() if v == 'partylist')
print(f'Constituency: {c_count}, Party list: {p_count}')

In [ ]:
# === Cell 5: Load Qwen2.5-VL Model ===
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f'Loading {MODEL_ID} (4-bit quantized)...')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Limit image resolution to prevent OOM
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,    # ~200K pixels
    max_pixels=1024 * 28 * 28,   # ~800K pixels (default 2048 is too much for T4)
)
print(f'Model loaded! VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

In [ ]:
# === Cell 6: Preprocessing + Batch Extraction ===
from qwen_vl_utils import process_vision_info
from PIL import Image, ImageEnhance, ImageFilter
import gc

THAI_TO_ARABIC = str.maketrans('\u0e50\u0e51\u0e52\u0e53\u0e54\u0e55\u0e56\u0e57\u0e58\u0e59', '0123456789')

def clean_vote(text):
    s = str(text).translate(THAI_TO_ARABIC)
    s = re.sub(r'[,.\'\"\s]', '', s)
    s = re.sub(r'[^0-9]', '', s)
    return s if s else '0'


# === Image Preprocessing ===
def preprocess_image(img_path, target_width=1024):
    """Preprocess scanned document image for better OCR.
    
    - Resize to optimal width (Qwen trained on ~1024px wide patches)
    - Convert to RGB
    - Enhance contrast + sharpness for scanned docs
    """
    img = Image.open(img_path)
    
    # Convert to RGB if needed
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    # Resize: keep aspect ratio, target width
    w, h = img.size
    if w > target_width:
        ratio = target_width / w
        new_h = int(h * ratio)
        img = img.resize((target_width, new_h), Image.LANCZOS)
    
    # Enhance for scanned documents
    img = ImageEnhance.Contrast(img).enhance(1.5)    # boost contrast
    img = ImageEnhance.Sharpness(img).enhance(2.0)    # sharpen text
    
    return img


# === Batch Inference ===
PAGE_PROMPT = """อ่านตารางคะแนนเลือกตั้งจากภาพนี้
ดึงข้อมูล ชื่อพรรค และ คะแนน ของทุกแถวในตาราง

ตอบแต่ละแถวเป็น:
ชื่อพรรค|คะแนน

ตัวอย่าง:
เพื่อไทย|12345
ประชาธิปัตย์|6789

กฎ: ใช้ตัวเลข Arabic (0-9), ไม่ต้องใส่ comma, ตอบเฉพาะข้อมูลในตาราง"""


def ocr_batch(img_paths, batch_size=3):
    """OCR multiple pages in a single batch for speed.
    
    Returns list of response strings, one per page.
    """
    all_responses = []
    
    for batch_start in range(0, len(img_paths), batch_size):
        batch_paths = img_paths[batch_start:batch_start + batch_size]
        
        # Preprocess images
        batch_images = []
        batch_messages = []
        batch_texts = []
        
        for img_path in batch_paths:
            pil_img = preprocess_image(img_path)
            batch_images.append(pil_img)
            
            msg = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": pil_img},
                    {"type": "text", "text": PAGE_PROMPT},
                ]
            }]
            batch_messages.append(msg)
            batch_texts.append(
                processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
            )
        
        # Process all images in batch
        all_image_inputs = []
        for msg in batch_messages:
            img_inputs, _ = process_vision_info(msg)
            all_image_inputs.extend(img_inputs)
        
        inputs = processor(
            text=batch_texts,
            images=all_image_inputs,
            padding=True,
            return_tensors="pt"
        ).to(model.device)
        
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=600,
                do_sample=False,
            )
        
        # Decode each response
        for j in range(len(batch_paths)):
            out_ids = generated_ids[j][inputs['input_ids'].shape[1]:]
            resp = processor.decode(out_ids, skip_special_tokens=True).strip()
            all_responses.append(resp)
        
        # Free memory
        del inputs, generated_ids, batch_images
        torch.cuda.empty_cache()
        gc.collect()
    
    return all_responses


def parse_ocr_response(response):
    """Parse pipe-delimited OCR response into dict."""
    pairs = {}
    for line in response.strip().split('\n'):
        line = line.strip().strip('- *')
        if '|' in line:
            parts = line.split('|', 1)
            if len(parts) >= 2:
                party = parts[0].strip()
                vote = clean_vote(parts[1])
                if party and vote != '0':
                    pairs[party] = vote
    return pairs


def extract_document(doc_id, page_paths, doc_type, expected_parties):
    """Extract votes: batch OCR pages, combine, fuzzy match to template."""
    
    img_paths = [p for _, p in page_paths]
    
    # Batch OCR all pages
    try:
        responses = ocr_batch(img_paths, batch_size=BATCH_SIZE)
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'    OOM with batch={BATCH_SIZE}, falling back to 1')
            torch.cuda.empty_cache()
            responses = ocr_batch(img_paths, batch_size=1)
        else:
            raise
    
    # Combine results from all pages
    all_pairs = {}
    for resp in responses:
        pairs = parse_ocr_response(resp)
        all_pairs.update(pairs)
    
    # Fuzzy match to expected parties
    results = []
    used_keys = set()
    
    for exp_party in expected_parties:
        best_match = None
        best_score = 0
        
        for ocr_party, vote in all_pairs.items():
            if ocr_party in used_keys:
                continue
            score = max(
                fuzz.ratio(exp_party, ocr_party),
                fuzz.partial_ratio(exp_party, ocr_party),
                fuzz.token_sort_ratio(exp_party, ocr_party),
            )
            if score > best_score and score >= 55:
                best_score = score
                best_match = (ocr_party, vote)
        
        if best_match:
            used_keys.add(best_match[0])
            results.append({
                'party_name': exp_party,
                'votes': best_match[1],
                'ocr_name': best_match[0],
                'score': best_score,
            })
        else:
            results.append({
                'party_name': exp_party,
                'votes': '0',
                'ocr_name': '',
                'score': 0,
            })
    
    return results


# Auto-detect batch size based on free VRAM
free_vram = (torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()) / 1024**3
if free_vram > 10:
    BATCH_SIZE = 4
elif free_vram > 7:
    BATCH_SIZE = 3
elif free_vram > 5:
    BATCH_SIZE = 2
else:
    BATCH_SIZE = 1

print(f'Extraction ready | Preprocessing: resize→1024px, contrast↑1.5, sharpness↑2.0')
print(f'Batch size: {BATCH_SIZE} (free VRAM: {free_vram:.1f} GB)')

In [ ]:
# === Cell 7: Test on sample document ===
test_doc = list(documents.keys())[0]
test_type = doc_types[test_doc]
expected_parties = [e['party_name'] for e in doc_expected[test_doc]]

print(f'Testing: {test_doc} ({test_type})')
print(f'Pages: {len(documents[test_doc])}, Expected parties: {len(expected_parties)}')
print()

t0 = time.time()
results_test = extract_document(test_doc, documents[test_doc], test_type, expected_parties)
elapsed = time.time() - t0

matched = sum(1 for r in results_test if r['votes'] != '0')
print(f'Extracted {matched}/{len(results_test)} parties in {elapsed:.1f}s:')
for r in results_test:
    ocr = r.get('ocr_name', '')[:20]
    score = r.get('score', 0)
    marker = '✓' if score >= 70 else '?' if score >= 55 else '✗'
    print(f"  {marker} {r['party_name']:<25s} → {r['votes']:>8s}  (score={score:3.0f} ocr='{ocr}')")

# Compare with sample label
label_file = SAMPLE_LABELS_DIR / f'{test_doc}.json'
if label_file.exists():
    with open(label_file, encoding='utf-8') as f:
        label = json.load(f)
    print(f'\n--- Ground Truth ---')
    label_map = {}
    for r in label.get('results', []):
        p = r.get('party', r.get('party_name', ''))
        label_map[p] = str(r['votes'])
    
    for r in results_test:
        actual = '?'
        for lp, lv in label_map.items():
            if fuzz.ratio(r['party_name'], lp) > 70:
                actual = lv
                break
        match = '✓' if actual == r['votes'] else '✗'
        print(f"  {match} {r['party_name']:<25s}  actual={actual:>8s}  pred={r['votes']:>8s}")

per_page = elapsed / len(documents[test_doc])
total_pages = sum(len(v) for v in documents.values())
print(f'\nSpeed: {per_page:.1f}s/page × {total_pages} pages = ~{per_page*total_pages/60:.0f} min total')

In [ ]:
# === Cell 8: Checkpoint & Save Functions ===

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


def save_submission_csv(results, template_df, doc_expected, output_dir):
    """Generate submission CSV."""
    submission_votes = {}
    for doc_id, expected_rows in doc_expected.items():
        ocr_rows = results.get(doc_id, [])
        for i, exp in enumerate(expected_rows):
            if i < len(ocr_rows):
                v = ocr_rows[i].get('votes', '0')
                submission_votes[exp['id']] = v if v else '0'
            else:
                submission_votes[exp['id']] = '0'
    
    sub = template_df[['id']].copy()
    sub['votes'] = sub['id'].map(submission_votes).fillna('0')
    csv_path = output_dir / 'submission.csv'
    sub.to_csv(csv_path, index=False)
    return csv_path


def update_progress(done, total, errors, start_time):
    elapsed = time.time() - start_time
    speed = done / elapsed if elapsed > 0 else 0
    eta = (total - done) / speed if speed > 0 else 0
    with open(PROGRESS_FILE, 'w') as f:
        f.write(f'OCR Progress\n')
        f.write(f'============\n')
        f.write(f'Completed: {done}/{total} ({done/total*100:.1f}%)\n')
        f.write(f'Errors: {len(errors)}\n')
        f.write(f'Speed: {speed:.2f} docs/sec ({1/speed:.1f} sec/doc)\n')
        f.write(f'Elapsed: {elapsed/60:.1f} min\n')
        f.write(f'ETA: {eta/60:.1f} min\n')
        f.write(f'Updated: {datetime.now().strftime("%H:%M:%S")}\n')

print('Save functions ready.')

In [ ]:
# === Cell 9: Main Processing Loop ===
results = load_checkpoint()
print(f'Loaded {len(results)} existing results from checkpoint.')

doc_ids_sorted = sorted(documents.keys(), key=lambda d: (doc_types.get(d, 'z'), d))
todo = [d for d in doc_ids_sorted if d not in results]
print(f'Documents to process: {len(todo)} (skipped {len(results)} already done)')

SAVE_EVERY = 5
errors = []
start_time = time.time()

for i, doc_id in enumerate(tqdm(todo, desc='OCR')):
    doc_type = doc_types.get(doc_id, 'constituency')
    page_paths = documents[doc_id]
    expected_parties = [e['party_name'] for e in doc_expected[doc_id]]
    
    try:
        rows = extract_document(doc_id, page_paths, doc_type, expected_parties)
        results[doc_id] = rows
        
        matched = sum(1 for r in rows if r.get('votes', '0') != '0')
        if matched < len(expected_parties) * 0.3:
            print(f'  WARN {doc_id}: only {matched}/{len(expected_parties)} parties found')
    
    except Exception as e:
        errors.append(doc_id)
        print(f'  ERROR {doc_id}: {e}')
        results[doc_id] = [{'party_name': p, 'votes': '0'} for p in expected_parties]
        torch.cuda.empty_cache()
    
    # Periodic save
    done = len(results)
    if done % SAVE_EVERY == 0 or i == len(todo) - 1:
        save_checkpoint(results)
        save_submission_csv(results, template_df, doc_expected, OUTPUT_DIR)
        update_progress(done, len(documents), errors, start_time)
        elapsed = time.time() - start_time
        speed = elapsed / done if done > 0 else 0
        eta = speed * (len(todo) - (i + 1))
        print(f'  [{done}/{len(documents)}] {elapsed/60:.1f}min, ETA {eta/60:.0f}min, CSV saved')

# Final save
save_checkpoint(results)
save_submission_csv(results, template_df, doc_expected, OUTPUT_DIR)

elapsed = time.time() - start_time
print(f'\nDone! {len(results)}/{len(documents)} docs in {elapsed/60:.1f} min')
if errors:
    print(f'Errors ({len(errors)}): {errors}')
print(f'Submission: {OUTPUT_DIR}/submission.csv')

In [ ]:
# === Cell 10: Validate Against Sample Labels ===

def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[j]+1, prev[j+1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]

results = load_checkpoint()

if SAMPLE_LABELS_DIR.exists():
    total_dist = 0
    total_pairs = 0
    
    for jf in sorted(SAMPLE_LABELS_DIR.glob('*.json')):
        with open(jf, encoding='utf-8') as f:
            label = json.load(f)
        doc_id = jf.stem
        label_results = label.get('results', [])
        expected_rows = doc_expected.get(doc_id, [])
        ocr_rows = results.get(doc_id, [])
        
        print(f'\n=== {doc_id} ===')
        
        for label_row in label_results:
            actual = str(label_row['votes'])
            party = label_row.get('party', label_row.get('party_name', ''))
            
            predicted = '0'
            for i, exp in enumerate(expected_rows):
                if fuzz.ratio(party, exp['party_name']) > 70:
                    if i < len(ocr_rows):
                        predicted = ocr_rows[i].get('votes', '0')
                    break
            
            dist = levenshtein(actual, predicted)
            total_dist += dist
            total_pairs += 1
            
            if dist > 0:
                print(f'  {party:30s}  actual={actual:>8s}  pred={predicted:>8s}  dist={dist}')
    
    print(f'\n=== Sample Validation ===')
    print(f'Pairs: {total_pairs}')
    print(f'Mean Levenshtein: {total_dist/total_pairs:.4f}')
else:
    print('No sample labels found.')

In [ ]:
# === Cell 11: Download Submission ===
from google.colab import files
files.download(str(OUTPUT_DIR / 'submission.csv'))